# Bronze Layer Notebook

Ingests raw JSON files from Unity Catalog Volumes into Delta Bronze tables.

## Notebook Goals
- Load raw extracts from volume paths.
- Normalize timestamp-like fields.
- Append data into Bronze Delta tables.
- Run basic data sanity checks.

## 1. Setup

Initialize dependencies and set the active catalog/schema for Bronze writes.

In [0]:
# Step 1: Import libraries
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

In [0]:
# Step 2: Configure catalog and schema context
# - Set active catalog to main
# - Set active schema to retail

spark.sql("USE CATALOG retail")
spark.sql("CREATE SCHEMA IF NOT EXISTS retail.bronze")
spark.sql("USE SCHEMA bronze")
# - Verify current catalog and schema
display(spark.sql("SELECT current_catalog(), current_schema()"))

## 2. Source Paths and Raw Ingestion

Define raw volume paths and load source JSON datasets for all entities.

In [0]:
# Step 3: Define raw source paths from Bronze volume
raw_base = "/Volumes/retail/raw/source_data"
customers_path = f"{raw_base}/customers"
order_items_path = f"{raw_base}/order_items"
orders_path = f"{raw_base}/orders"
products_path = f"{raw_base}/products"
watermarks_path = f"{raw_base}/watermarks.json"


In [0]:
# Step 4: Read raw JSONL for each table and display a sample
customers_raw_df = spark.read.json(customers_path)
products_raw_df = spark.read.json(products_path)
orders_raw_df = spark.read.json(orders_path)
order_items_raw_df = spark.read.json(order_items_path)

display(customers_raw_df)
display(products_raw_df)
display(orders_raw_df)
display(order_items_raw_df)

In [0]:
# Step 5: Normalize _extracted_at, created_at, updated_at columns to timestamp in all DataFrames

def normalize_timestamps(df):
    for col in ['_extracted_at', 'created_at', 'updated_at', 'order_ts']:
        if col in df.columns:
            df = df.withColumn(col, F.to_timestamp(col))
    return df

customers_raw_df = normalize_timestamps(customers_raw_df)
products_raw_df = normalize_timestamps(products_raw_df)
orders_raw_df = normalize_timestamps(orders_raw_df)
order_items_raw_df = normalize_timestamps(order_items_raw_df)

In [0]:
display(customers_raw_df)
display(products_raw_df)
display(orders_raw_df)
display(order_items_raw_df)

## 3. Bronze Table Loads

Append the prepared datasets into Bronze Delta tables.

In [0]:
# Step 7: Write Bronze Delta tables
# - Write table bronze.customers
customers_raw_df.write.format("delta").mode("append").saveAsTable("bronze.customers")
# - Write table bronze.products
products_raw_df.write.format("delta").mode("append").saveAsTable("bronze.products")
# - Write table bronze.orders
orders_raw_df.write.format("delta").mode("append").saveAsTable("bronze.orders")
# - Write table bronze.order_items
order_items_raw_df.write.format("delta").mode("append").saveAsTable("bronze.order_items")

## 4. Validation Checks

Run row-level inspections and primary key null checks on Bronze tables.

In [0]:
# Step 8: Validate Bronze outputs

# Show 5 sample rows from each bronze table
display(spark.sql("SELECT * FROM customers LIMIT 5"))
display(spark.sql("SELECT * FROM products LIMIT 5"))
display(spark.sql("SELECT * FROM orders LIMIT 5"))
display(spark.sql("SELECT * FROM order_items LIMIT 5"))

# Check nulls in primary keys
display(spark.sql("SELECT COUNT(*) AS null_count FROM customers WHERE customer_id IS NULL"))
display(spark.sql("SELECT COUNT(*) AS null_count FROM products WHERE product_id IS NULL"))
display(spark.sql("SELECT COUNT(*) AS null_count FROM orders WHERE order_id IS NULL"))
display(spark.sql("SELECT COUNT(*) AS null_count FROM order_items WHERE order_item_id IS NULL"))



## 5. Source-to-Bronze Reconciliation

Compare source file counts against Bronze table counts and capture final notes for interview discussion.

In [0]:
# Compare source-to-bronze counts
display(spark.sql(f"SELECT 'customers' AS table, (SELECT COUNT(*) FROM customers) AS bronze_count, (SELECT COUNT(*) FROM json.`{customers_path}`) AS source_count"))
display(spark.sql(f"SELECT 'products' AS table, (SELECT COUNT(*) FROM products) AS bronze_count, (SELECT COUNT(*) FROM json.`{products_path}`) AS source_count"))
display(spark.sql(f"SELECT 'orders' AS table, (SELECT COUNT(*) FROM orders) AS bronze_count, (SELECT COUNT(*) FROM json.`{orders_path}`) AS source_count"))
display(spark.sql(f"SELECT 'order_items' AS table, (SELECT COUNT(*) FROM order_items) AS bronze_count, (SELECT COUNT(*) FROM json.`{order_items_path}`) AS source_count"))

## 6. Final Notes

Capture implementation decisions, limitations, and handoff notes for the Silver layer.